## 1. Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [2]:
df = pd.read_csv('data/students_dropout_dataset.csv',sep=';')
y_target = 'Target'
df.head(5)

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 2. Handle Missing / Invalid Values

In [3]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 0
Missing Feature:
[]


## 3. Analisis & Handling Outliers

In [4]:
feature_numerik = ['Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)','Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)','Age at enrollment',
                   'Admission grade','Previous qualification (grade)']

Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 1225
Jumlah Outlier Sesudah Dihapus: 0


## 4. Feature Engineering

In [5]:
kondisi = df['Curricular units 1st sem (enrolled)'] > 0
df['Approval Rate 1st sem'] = np.where(kondisi, df['Curricular units 1st sem (approved)'] / df['Curricular units 1st sem (enrolled)'], 0)
df['Evaluation Rate 1st sem'] = np.where(kondisi, df['Curricular units 1st sem (evaluations)'] / df['Curricular units 1st sem (enrolled)'], 0)

df['financial_stress_score'] = df['Debtor'] + (1 - df['Tuition fees up to date']) + (1 - df['Scholarship holder'])
df['grade_diff_admission_vs_prev'] = df['Admission grade'] - df['Previous qualification (grade)']

## 5.Save Cleaned Dataset

In [6]:
file_path = 'data/students_dropout_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')
df.head()

File belum ada. Berhasil menyimpan dataset baru!


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target,Approval Rate 1st sem,Evaluation Rate 1st sem,financial_stress_score,grade_diff_admission_vs_prev
1,1,15,1,9254,1,1,160.0,1,1,3,...,13.666667,0,13.9,-0.3,0.79,Graduate,1.0,1.000000,2,-17.5
2,1,1,5,9070,1,1,122.0,1,37,37,...,0.000000,0,10.8,1.4,1.74,Dropout,0.0,0.000000,2,2.8
3,1,17,2,9773,1,1,122.0,1,38,37,...,12.400000,0,9.4,-0.8,-3.12,Graduate,1.0,1.333333,1,-2.4
6,1,1,1,9500,1,1,142.0,1,19,38,...,14.345000,0,15.5,2.8,-4.06,Graduate,1.0,1.285714,0,-13.6
7,1,18,4,9254,1,1,119.0,1,37,37,...,0.000000,0,15.5,2.8,-4.06,Dropout,0.0,1.000000,2,-5.9
